# v1
The only thing we see is that at layers.2, out12, in6, (2,4), we always have a `| |` alignment, which is detecting the pattern of empty space between two diagonal `/` lines.   

Sometimes it is spurious, but this suggests that there is high spatial correlation in the data about this point.  
We might see more examples of these points in this kernel around this space.  
Maybe in the area from [2:4, 3:5], I can check more points around that space to see if we find more.   
It might be interesting to actually see how this slice came about though.  

I should write down the propositions for one of them.  
Somehow.  


# Proposition chain
For the URL:
http://localhost:5173/models/simple_mnist_v1/4-oVe/v1/kernel-slice/layers.2/12/6


- Input pixels has some black parts and some green parts.  
- Its best to not write propositions spatially as it might hurt my judgement, I dont really want to see what the set of propositions looks like in my head.  

- Each pixel carries a proposition, the proposition talks about how its patch was aligned for now.  

Main POI: `{layers.2, out.12, in.6}[2,4]`
Receptive field: `{layers.1, out.6}[3:6, 7:10]`
Receptive field: `{layers.0, out.6}[3:6, 7:10]`

Receptive field of the layers.0 patch (only one input channel haash)
The final receptive field: `{x, out.0}[5:11, 13:19]`


## {layers.0} propositions

Two labels: `all red`, `some green`   
Two output labels: `green`, `red`

```js
// the source vs target pattern, this is also important
[3,7]: (all red -> green)
[3,8]: (all red -> green)
[3,9]: (all red -> green)
[4,7]: (some green -> red)
[4,8]: (all red -> green)
[4,9]: (some green -> red)
[5,7]: (some green -> red)
[5,8]: (all red -> green)
[5,9]: (some green -> red)
```

We can write the full pixel as `{layers.0, out.6}[3,7](all red -> green)`

## {layers.1} propositions
```js
// the source vs target pattern, this is also important
[3,7]: (green -> green)
[3,8]: (green -> green)
[3,9]: (green -> green)
[4,7]: (red -> black)
[4,8]: (green -> green)
[4,9]: (red -> black)
[5,7]: (red -> black)
[5,8]: (green -> green)
[5,9]: (red -> black)
```
This is a pointwise operation only, for our case, the propositions dont change much.  
They do see some destruction in general from ReLU though (all red information is empty for the network, other than patterns though, basically the network can't reason about the reds from magnitude, it can use spatial patterns though).  
For this case, it does not matter much.   

It would later be useful to see these as 1d arrays, so that i can see the propositions without seeing the image.  

## {layers.2} propositions


Now the interesting part is that there is some pattern the kernel is using to determine the uniqueness of the output given by the final layer.  
This is the first question, can the final layer actually differentiate between the multiple patterns defined by a kernel?  
It might be useful to see the weight corresponding to this (the flattened weight of the last layer, unflattens to our channel, lets see (although, it is dependent on the cube and not the slice that im thinking right now)).  

Let us assume that we cannot differentiate from the input.  
In this case, the final pattern actually is forming a diagonal, from the kernels final state, i can only say that the green color means:

- `/` or a `| |` (there is however, a high correlation of finding `| |` at (2,4) lol, is this a spatial thing?).  
- The final weight is really just a multiplication, lets see how it goes.  

For this kernel however, there is only:
- perfect alignment which -> `| |`
- or `/` for some range

Lets verify this.  



# v2

From what i see, the kernel's output is basically more dependent on detecting `/`.  
For the POI (2,4), it consistently lies between the two vertical parts of 4, and as such has a higher chance of making the `| |` pattern.  
This is not bad, but I don't see any evidence of being able to differentiate between the two from the output activation alone (at least, from the point (2,4) it is hard to see a correlation).   
It might be that it can look at the surrounding, i haven't checked that out. The surroundings have lesser contribs though (that might be a problem of the contrib finder too).  

In any case, due to this uncertainity, im only going to assert that we found at least a `/` on the left for `| |` for now and try to frame the proposition.  

I might be shooting a dud here i think, is there something to see?  
Can I frame it?  


Lets start by how I would see automatic labelling.  
`{layers.2, o12, i6}[2,4]` is green. this means that we found black pixels on left column, green on right.  

Receptive field 1:
`{layers.1, o6}[3:6, 7:10]`. Out of this, so we say:
```js
{layers.1, o6}
  [3, 7]: black
  [4, 7]: black
  [5, 7]: black

  [3, 8]: green
  [4, 8]: green
  [5, 8]: green

  [3, 9]: green
  [4, 9]: green
  [5, 9]: green
```

This is what the output being green there is telling me right now.  

Now from {layers.1}, we have the reverse propositions  
```js
black -> red
green -> green
```
This gives us
```js
// proposition backpropped or something lol
{layers.0, o6}
  [3, 7]: red
  [4, 7]: red
  [5, 7]: red

  [3, 8]: green
  [4, 8]: green
  [5, 8]: green

  [3, 9]: green
  [4, 9]: green
  [5, 9]: green
```

Now the interesting part, the receptive field is going to increase for each and we are going to talk about their meanings.  

In {layers.0}, the kernel gives a red if there are some greens inside the patch, else it gives a green.  (all reds)
```js
{layers.0, o6}
[3,7] = red (means) receptive field [5:8,13:16] has some green
[4,7] = red (means) receptive field [7:10,13:16] has some green
[5,7] = red (means) receptive field [9:12,13:16] has some green

[3, 8]: green (means) receptive field [5:8, 15:18] has all red
[4, 8]: green (means) receptive field [7:10, 15:18] has all red
[5, 8]: green (means) receptive field [9:12, 15:18] has all red

[3, 9]: green (means) receptive field [5:8, 17:20] has all red
[4, 9]: green (means) receptive field [7:10, 17:20] has all red
[5, 9]: green (means) receptive field [9:12, 17:20] has all red
```

And thats it, it does not say whether those some greens make a vertical edge, or whatever.  
But since the upper kernel is finding that left column was red, there is a vertical edge now (green on left, red on right).  

Now this thing, is not much lol.  
But I might need a better notation who knows, the interesting parts come when it composes in the larger network, maybe this meaning thing is not that useful, but composition can help to explain the story better.  
It makes intuitive sense to me that the last receptive field is a (green on left, red on right border with x=15:20 being fully red in the patch).  
There is one thing though, since the final kernel also has a chance that it found a green on its right, the last receptive patch [17:20] also has a chance of having some green. That changes its proposition, for now, though, this is the only information i can deduce.  

There is a probablity to what pattern the kernel detects, its easy to calculate that since we have the training set, but is there a relation to how the propositions are modeled?  


Moving on, later I would also try to use this framework for inception and see if im able to come up with something (we already have the distribution of the pattern emerging in the training data, it might be useful to add that proposition to this somehow).  
I don't know how they are composing across layers though, im fully lost on this.  

I would later need to do some reading, but this is not the most pressing concern right now.  
Although, it would interesting to say probabilistically what kind of input would be generated from this.  